<a href="https://colab.research.google.com/github/gnagacharan999-sudo/my-colab-project/blob/main/smart_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1: Imports, Colab Secret loading, API configuration
# ============================================================
!pip install -q gradio pypdf requests

import re
import json
import requests
import gradio as gr
from pypdf import PdfReader
from collections import Counter

SERPAPI_URL = "https://serpapi.com/search"
REQUEST_TIMEOUT = 30


def load_serpapi_key():
    """Read the SerpApi key from Colab Secrets (never typed in the UI, never printed)."""
    try:
        from google.colab import userdata
        key = userdata.get("SERPAPI_KEY")
        return key.strip() if key else None
    except Exception as e:
        print("Could not read SERPAPI_KEY from Colab Secrets:", type(e).__name__)
        return None


SERPAPI_KEY = load_serpapi_key()

if SERPAPI_KEY:
    print("✅ SerpApi key loaded from Colab Secrets.")
else:
    print("⚠️ SERPAPI_KEY not found. Click the 🔑 icon in Colab, add a secret named SERPAPI_KEY,")
    print("   turn on 'Notebook access', then run this cell again.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 2.6 MB/s eta 0:00:00
✅ SerpApi key loaded from Colab Secrets.


In [ ]:
# ============================================================
# CELL 2: SKILL_LIST, COURSE_MAP and supporting dictionaries
# ============================================================
SKILL_LIST = {
    "python", "java", "c", "c++", "sql", "mysql", "mongodb", "html", "css",
    "javascript", "react", "node.js", "git", "github", "machine learning",
    "deep learning", "artificial intelligence", "data science", "pandas",
    "numpy", "matplotlib", "scikit-learn", "tensorflow", "pytorch", "flask",
    "django", "fastapi", "power bi", "tableau", "excel", "aws", "azure",
    "docker", "linux", "rest api", "api", "data structures", "algorithms",
    "oops", "dbms", "operating systems", "computer networks"
}

# Alternative spellings that should count as the same skill
SKILL_ALIASES = {
    "nodejs": "node.js", "node js": "node.js", "node": "node.js",
    "scikit learn": "scikit-learn", "sklearn": "scikit-learn",
    "oop": "oops", "object oriented programming": "oops",
    "object-oriented programming": "oops",
    "ml": "machine learning", "dl": "deep learning",
    "ai": "artificial intelligence", "js": "javascript", "cpp": "c++",
    "restful api": "rest api", "restful apis": "rest api", "rest apis": "rest api",
    "ms excel": "excel", "powerbi": "power bi", "os": "operating systems",
}

COURSE_MAP = {
    "python": "Python Programming",
    "java": "Object-Oriented Programming with Java",
    "c": "C Programming",
    "c++": "C++ and Object-Oriented Programming",
    "sql": "DBMS and SQL",
    "mysql": "DBMS and SQL",
    "mongodb": "Database Management Systems",
    "html": "Web Technologies",
    "css": "Web Technologies",
    "javascript": "Web Technologies",
    "react": "Web Application Development",
    "node.js": "Web Application Development",
    "machine learning": "Machine Learning",
    "deep learning": "Deep Learning",
    "artificial intelligence": "Artificial Intelligence",
    "data science": "Data Science",
    "pandas": "Data Analytics using Python",
    "numpy": "Numerical Computing with Python",
    "scikit-learn": "Machine Learning",
    "tensorflow": "Deep Learning",
    "pytorch": "Deep Learning",
    "git": "Software Development Tools",
    "github": "Software Development Tools",
    "rest api": "Computer Networks / Web Services",
    "api": "Web Services and API Integration",
    "data structures": "Data Structures",
    "algorithms": "Design and Analysis of Algorithms",
    "oops": "Object-Oriented Programming",
    "dbms": "Database Management Systems",
    "operating systems": "Operating Systems",
    "computer networks": "Computer Networks",
    # extra skills from SKILL_LIST that were not in the original map
    "flask": "Web Application Development",
    "django": "Web Application Development",
    "fastapi": "Web Services and API Development",
    "matplotlib": "Data Visualization with Python",
    "power bi": "Data Analytics and Visualization",
    "tableau": "Data Analytics and Visualization",
    "excel": "Data Analytics and Visualization",
    "aws": "Cloud Computing",
    "azure": "Cloud Computing",
    "docker": "Software Development Tools / DevOps",
    "linux": "Operating Systems",
}

# Used to organise the Skills section suggestion
SKILL_CATEGORIES = {
    "Programming Languages": ["python", "java", "c", "c++", "javascript"],
    "Web Technologies": ["html", "css", "react", "node.js", "flask", "django", "fastapi", "rest api", "api"],
    "Databases": ["sql", "mysql", "mongodb", "dbms"],
    "AI / ML & Data": ["machine learning", "deep learning", "artificial intelligence", "data science",
                       "pandas", "numpy", "matplotlib", "scikit-learn", "tensorflow", "pytorch",
                       "power bi", "tableau", "excel"],
    "Tools & Cloud": ["git", "github", "docker", "linux", "aws", "azure"],
    "CS Fundamentals": ["data structures", "algorithms", "oops", "operating systems", "computer networks"],
}

DISPLAY_NAMES = {
    "sql": "SQL", "mysql": "MySQL", "mongodb": "MongoDB", "html": "HTML", "css": "CSS",
    "javascript": "JavaScript", "node.js": "Node.js", "github": "GitHub", "aws": "AWS",
    "api": "API", "rest api": "REST API", "dbms": "DBMS", "oops": "OOPs", "c": "C",
    "c++": "C++", "fastapi": "FastAPI", "pytorch": "PyTorch", "tensorflow": "TensorFlow",
    "numpy": "NumPy", "scikit-learn": "scikit-learn", "power bi": "Power BI",
}


def display_skill(skill):
    """Pretty name for a skill, e.g. 'sql' -> 'SQL'."""
    return DISPLAY_NAMES.get(skill, skill.title())


print(f"✅ Loaded {len(SKILL_LIST)} skills and {len(COURSE_MAP)} course mappings.")

✅ Loaded 42 skills and 42 course mappings.


In [ ]:
# ============================================================
# CELL 3: Resume extraction, skill detection, manual skill parsing
# ============================================================
def extract_resume_text(file_path):
    """Extract text from every page of a PDF. Returns '' if it fails or is image-only."""
    if not file_path:
        return ""
    if not isinstance(file_path, str):
        file_path = getattr(file_path, "name", "")
    try:
        reader = PdfReader(file_path)
        pages = []
        for page in reader.pages:
            pages.append(page.extract_text() or "")
        return "\n".join(pages)
    except Exception:
        return ""


def _make_pattern(term):
    # Whole-term match so "c" does not match inside "c++", and "git" not inside "github"
    return re.compile(r"(?<![a-z0-9+#])" + re.escape(term) + r"(?![a-z0-9+#])", re.IGNORECASE)


SKILL_PATTERNS = {skill: _make_pattern(skill) for skill in SKILL_LIST}
ALIAS_PATTERNS = [(_make_pattern(alias), skill) for alias, skill in SKILL_ALIASES.items()]


def detect_skills(text):
    """Find skills from SKILL_LIST in any text (case-insensitive). Returns a sorted list."""
    if not text:
        return []
    text = re.sub(r"\s+", " ", str(text))
    found = set()
    for skill, pattern in SKILL_PATTERNS.items():
        if pattern.search(text):
            found.add(skill)
    for pattern, skill in ALIAS_PATTERNS:
        if pattern.search(text):
            found.add(skill)
    return sorted(found)


def parse_manual_skills(text):
    """Parse comma / semicolon / newline separated skills typed by the user."""
    if not text:
        return []
    skills = []
    for item in re.split(r"[,;\n]+", str(text)):
        skill = item.strip().lstrip("-•* ").strip().lower()
        if not skill:
            continue
        skill = SKILL_ALIASES.get(skill, skill)
        if skill not in skills:
            skills.append(skill)
    return skills


def detect_education(text):
    """Best-effort education line from a resume."""
    m = re.search(r"(b\.?\s?tech|b\.?\s?e\b|m\.?\s?tech|bca|mca|b\.?\s?sc|m\.?\s?sc|diploma|mba)[^\n]{0,60}",
                  text or "", re.IGNORECASE)
    return m.group(0).strip() if m else "Not specified"


print("✅ Resume and skill functions ready.")

✅ Resume and skill functions ready.


In [ ]:
# ============================================================
# CELL 4: SerpApi Google Jobs -> get_jobs()
# ============================================================
from concurrent.futures import ThreadPoolExecutor
import html as html_lib

PAGE_TIMEOUT = 6          # seconds allowed per employer page when reading experience
BARE_RANGE = re.compile(r"(\d{1,2})\s*(?:to|-|–)\s*(\d{1,2})\s*\+?\s*(?:yrs?|years?)\b", re.IGNORECASE)

JOB_TYPE_CHIPS = {"Full Time": "FULLTIME", "Part Time": "PARTTIME",
                  "Internship": "INTERN", "Contract": "CONTRACTOR"}
JOB_TYPE_WORDS = {"Full Time": "full time", "Part Time": "part time",
                  "Internship": "internship", "Contract": "contract"}

JOB_TYPE_KEYS = {"Full Time": "full", "Part Time": "part", "Internship": "intern", "Contract": "contract"}

SENIOR_TITLE = re.compile(r"\b(senior|sr|lead|principal|staff|architect|manager|head|director|vp|vice president)\b",
                          re.IGNORECASE)
EXP_PATTERN = re.compile(r"(\d{1,2})(?:\s*(?:-|–|to)\s*(\d{1,2}))?\s*\+?\s*(?:years?|yrs?|yr)\b", re.IGNORECASE)
MONTHS_PATTERN = re.compile(r"(\d{1,2})\s*\+?\s*months?", re.IGNORECASE)
FRESHER_PATTERN = re.compile(r"\b(fresher|freshers|entry[- ]level|no experience|graduate trainee|recent graduates?|0\s*(?:-|to)\s*1 years?)\b", re.IGNORECASE)

LOCATION_ALIASES = {
    "bangalore": ["bangalore", "bengaluru"], "bengaluru": ["bangalore", "bengaluru"],
    "gurgaon": ["gurgaon", "gurugram"], "gurugram": ["gurgaon", "gurugram"],
    "mumbai": ["mumbai", "bombay"], "bombay": ["mumbai", "bombay"],
    "chennai": ["chennai", "madras"], "madras": ["chennai", "madras"],
    "kolkata": ["kolkata", "calcutta"], "calcutta": ["kolkata", "calcutta"],
    "visakhapatnam": ["visakhapatnam", "vizag"], "vizag": ["visakhapatnam", "vizag"],
    "delhi": ["delhi"], "new delhi": ["delhi"],
    "hyderabad": ["hyderabad", "secunderabad"], "secunderabad": ["hyderabad", "secunderabad"],
}

NOT_STATED = "Not stated in listing"
UNKNOWN_LABELS = {"not stated in listing", "not specified", ""}

INDIA_HINTS = [
    "india", "hyderabad", "telangana", "bengaluru", "bangalore", "karnataka", "chennai",
    "tamil nadu", "mumbai", "maharashtra", "pune", "delhi", "noida", "gurgaon", "gurugram",
    "haryana", "uttar pradesh", "kolkata", "west bengal", "kerala", "kochi", "gujarat",
    "ahmedabad", "andhra pradesh", "visakhapatnam", "rajasthan", "jaipur", "chandigarh",
    "coimbatore", "indore", "odisha", "bhubaneswar", "mysuru", "mysore", "nagpur", "vijayawada",
]


def clean_text(value, default="Not specified"):
    if value is None:
        return default
    text = str(value).strip()
    return text if text else default


def job_text(job):
    """All text of a job used for skill extraction."""
    return " ".join([job.get("title", ""), job.get("job_description", ""),
                     job.get("education_and_skills", ""), job.get("role_and_responsibility", "")])


def location_terms(location):
    city = location.split(",")[0].strip().lower()
    return LOCATION_ALIASES.get(city, [city])


def location_ok(job_location, city_terms, strict_city):
    loc = (job_location or "").lower()
    if not loc or loc == "not specified":
        return False
    if strict_city:
        return any(term in loc for term in city_terms)
    return any(hint in loc for hint in INDIA_HINTS)


def serpapi_request(params):
    """Call SerpApi safely. Returns (data, error_message)."""
    try:
        resp = requests.get(SERPAPI_URL, params=params, timeout=REQUEST_TIMEOUT)
    except requests.exceptions.Timeout:
        return None, "⏱️ The request to SerpApi timed out. Please try again."
    except requests.exceptions.RequestException:
        return None, "🌐 Network error while contacting SerpApi. Check your internet connection."

    try:
        data = resp.json()
    except ValueError:
        return None, "⚠️ SerpApi returned an invalid (non-JSON) response. Please try again."

    if not isinstance(data, dict):
        return None, "⚠️ SerpApi returned an unexpected response format."

    err = str(data.get("error", "")).strip()
    low = err.lower()

    if resp.status_code in (401, 403) or "invalid api key" in low or "api key" in low and "invalid" in low:
        return None, "🔑 SerpApi rejected the API key (invalid or expired). Update SERPAPI_KEY in Colab Secrets."
    if resp.status_code == 429 or "run out of searches" in low or "rate limit" in low or "quota" in low:
        return None, "📉 SerpApi quota or rate limit reached. Wait a while or check your SerpApi plan."
    if err:
        if "hasn't returned any results" in low or "no results" in low:
            return {"jobs_results": []}, None
        return None, f"⚠️ SerpApi error: {err}"
    if resp.status_code != 200:
        return None, f"⚠️ SerpApi returned HTTP status {resp.status_code}."
    return data, None


def search_page(params):
    """One search call, with fallbacks for an unsupported location or an over-strict job-type filter."""
    data, err = serpapi_request(params)
    if err and "location" in err.lower() and "location" in params:
        params.pop("location")
        data, err = serpapi_request(params)
    if err is None and not data.get("jobs_results") and "chips" in params:
        params.pop("chips")
        data2, err2 = serpapi_request(params)
        if err2 is None:
            data = data2
    return data, err


def get_highlight(raw, keywords):
    items = []
    for block in raw.get("job_highlights") or []:
        if isinstance(block, dict) and any(k in str(block.get("title", "")).lower() for k in keywords):
            items.extend(str(i) for i in (block.get("items") or []))
    return " ".join(items)


def get_apply_link(raw):
    """Only a real link from SerpApi's apply_options. Never invented."""
    for opt in raw.get("apply_options") or []:
        if isinstance(opt, dict):
            link = str(opt.get("link", "")).strip()
            if link.startswith("http"):
                return link
    return ""


def detect_job_type(raw, title, description=""):
    """1) SerpApi's schedule_type  2) the extensions list  3) the title  4) only CLEAR phrases in the description
    (loose mentions such as 'our internship program' are ignored to avoid wrong labels)."""
    sched = (raw.get("detected_extensions") or {}).get("schedule_type")
    if sched:
        return clean_text(sched)
    for ext in raw.get("extensions") or []:           # e.g. ["Full-time", "Posted 3 days ago"]
        e = str(ext).lower()
        if re.search(r"\bintern(ship)?s?\b", e):
            return "Internship"
        if re.search(r"\bfull[- ]time\b", e):
            return "Full-time"
        if re.search(r"\bpart[- ]time\b", e):
            return "Part-time"
        if re.search(r"\bcontract(or)?\b", e):
            return "Contract"
    low = title.lower()
    if re.search(r"\bintern(ship)?s?\b", low):
        return "Internship"
    if re.search(r"\bfull[- ]time\b", low):
        return "Full-time"
    if re.search(r"\bpart[- ]time\b", low):
        return "Part-time"
    if re.search(r"\bcontract(or)?\b", low):
        return "Contract"
    d = description.lower()
    clear = [
        ("Internship", r"(job|employment|position|role)\s*type\s*[:\-]?\s*intern|this is an? (paid |unpaid )?internship|internship (position|opportunity|role)"),
        ("Full-time", r"(job|employment|position|role)\s*type\s*[:\-]?\s*full[- ]time|full[- ]time (position|role|job|opportunity|employee)|this is a full[- ]time"),
        ("Part-time", r"(job|employment|position|role)\s*type\s*[:\-]?\s*part[- ]time|part[- ]time (position|role|job|opportunity)|this is a part[- ]time"),
        ("Contract", r"(job|employment|position|role)\s*type\s*[:\-]?\s*contract|contract(ual)? (position|role|basis)|on a contract basis"),
    ]
    for label, pattern in clear:
        if re.search(pattern, d):
            return label
    return NOT_STATED


def find_experience_requirements(text):
    """Return (low, high) year requirements written next to the word 'experience' / 'exp'."""
    found = []
    for m in EXP_PATTERN.finditer(text):
        window = text[max(0, m.start() - 60): m.end() + 60].lower()
        if re.search(r"experience|\bexp\b|experienced", window):
            low = int(m.group(1))
            high = int(m.group(2)) if m.group(2) else None
            if low <= 30:
                found.append((low, high))
    for m in MONTHS_PATTERN.finditer(text):          # e.g. "6 months experience" -> 0 years
        window = text[max(0, m.start() - 40): m.end() + 40].lower()
        if re.search(r"experience|\bexp\b", window) and int(m.group(1)) < 12:
            found.append((0, 1))
    return found


def min_required_years(text):
    """Minimum years of experience a job asks for. 0 = fresher friendly, None = not stated."""
    reqs = find_experience_requirements(text)
    if reqs:
        return max(low for low, _ in reqs)
    if FRESHER_PATTERN.search(text):
        return 0
    return None


def extract_experience(text):
    """Human-readable experience requirement for the job card."""
    reqs = find_experience_requirements(text)
    if reqs:
        low, high = max(reqs)
        return f"{low}-{high} years" if high else f"{low}+ years"
    if FRESHER_PATTERN.search(text):
        return "Fresher / Entry level"
    return "Not stated in listing"


def parse_user_experience(text):
    """Convert what the user typed ('Fresher', '1 year', '6 months') into years. None = no filter."""
    t = (text or "").strip().lower()
    if not t:
        return None
    if re.search(r"fresher|no experience|entry|student|none|nil|not applicable", t):
        return 0.0
    m = re.search(r"(\d+(?:\.\d+)?)", t)
    if m:
        years = float(m.group(1))
        return years / 12 if "month" in t else years
    return None


def job_type_match(job, job_type):
    """'yes' = matches, 'no' = clearly a different type, 'unknown' = type not stated by the employer."""
    key = JOB_TYPE_KEYS.get(job_type)
    if not key:
        return "yes"                                   # "Any"
    jt = job["job_type"].lower()
    title = job["title"].lower()
    is_intern_title = bool(re.search(r"\bintern(ship)?s?\b", title))
    if key == "intern":
        if "intern" in jt or is_intern_title:
            return "yes"
        return "unknown" if jt in UNKNOWN_LABELS else "no"
    if is_intern_title or "intern" in jt:
        return "no"
    if jt in UNKNOWN_LABELS:
        return "yes" if key in title else "unknown"
    return "yes" if key in jt else "no"


def experience_ok(job, user_years):
    """Remove jobs that need more experience than the user has."""
    if user_years is None:
        return True
    if user_years <= 2 and SENIOR_TITLE.search(job["title"]):
        return False
    required = job.get("min_experience")
    if required is None:
        return True
    allowed = user_years + (0 if user_years == 0 else 1)   # fresher: only 0-year jobs
    return required <= allowed


def normalize_job(raw):
    """Convert one SerpApi result into our standard job dictionary."""
    description = clean_text(raw.get("description"))
    edu_skills = get_highlight(raw, ["qualification", "skill", "education", "requirement"])
    role_resp = get_highlight(raw, ["responsibilit", "role", "duties"])
    all_text = f"{raw.get('title', '')} {description} {edu_skills}"
    via = clean_text(raw.get("via"), "")
    via = re.sub(r"^via\s+", "", via, flags=re.IGNORECASE) or "Not specified"
    return {
        "title": clean_text(raw.get("title")),
        "company": clean_text(raw.get("company_name")),
        "location": clean_text(raw.get("location")),
        "job_type": detect_job_type(raw, clean_text(raw.get("title"), ""), description),
        "experience": extract_experience(all_text),
        "min_experience": min_required_years(all_text),
        "job_description": description,
        "education_and_skills": edu_skills or "Not specified",
        "role_and_responsibility": role_resp or "Not specified",
        "apply_link": get_apply_link(raw),
        "via": via,
        "source_link": clean_text(raw.get("share_link"), ""),
    }


def fetch_experience_from_page(url):
    """Best effort: open the employer/job-board page and read an experience range such as '0 to 4 Yrs'.
    Returns (low, high) or None. Pages that block bots or need JavaScript simply return None."""
    try:
        resp = requests.get(url, timeout=PAGE_TIMEOUT,
                            headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                                                   "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"})
        if resp.status_code != 200:
            return None
        page = resp.text[:400000]
    except Exception:
        return None
    page = re.sub(r"(?is)<(script|style|noscript)[^>]*>.*?</\1>", " ", page)
    text = html_lib.unescape(re.sub(r"(?s)<[^>]+>", " ", page))
    text = re.sub(r"\s+", " ", text)
    m = BARE_RANGE.search(text[:4000])       # FIRST range on the page = the job's own highlights box, e.g. '0 to 4 Yrs'
    if m:                                    # (later ranges usually belong to 'Similar Jobs')
        return int(m.group(1)), int(m.group(2))
    reqs = find_experience_requirements(text[:3000])          # otherwise: wording with the word 'experience'
    if reqs:
        return reqs[0]
    return None


def enrich_experience(jobs):
    """For jobs whose experience is not in Google's text, read it from the apply page (in parallel)."""
    targets = [j for j in jobs if j.get("min_experience") is None and j.get("apply_link")]
    if not targets:
        return 0
    try:
        with ThreadPoolExecutor(max_workers=10) as pool:
            results = list(pool.map(lambda j: fetch_experience_from_page(j["apply_link"]), targets))
    except Exception:
        return 0
    found = 0
    for job, res in zip(targets, results):
        if res:
            low, high = res
            job["min_experience"] = low
            job["experience"] = (f"{low}-{high} years" if high is not None else f"{low}+ years") + " (from employer page)"
            found += 1
    return found


def relevance_score(job, role, user_skills):
    tokens = [t for t in re.findall(r"[a-z0-9+#.]+", role.lower()) if t not in {"and", "of", "the", "a", "in"}]
    title = job["title"].lower()
    desc = job_text(job).lower()
    score = sum(3 for t in tokens if t in title) + sum(1 for t in tokens if t in desc)
    score += 0.5 * len(set(user_skills) & set(detect_skills(job_text(job))))
    return score


def get_jobs(role, location, location_pref, job_type, user_skills=None, experience="", max_jobs=10, max_pages=3):
    """Search Google Jobs through SerpApi. Returns (list_of_jobs, message)."""
    if not SERPAPI_KEY:
        return [], ("❌ SerpApi key not found. Add SERPAPI_KEY in Colab Secrets (🔑 icon), "
                    "enable Notebook access and re-run Cell 1.")
    role = (role or "").strip()
    location = (location or "").strip()
    if not role:
        return [], "⚠️ Please enter a target job role."

    notes = []
    strict_city = location_pref == "Preferred Location Only"
    if strict_city and not location:
        strict_city = False
        notes.append("No location was entered, so the whole of India was searched.")
    city_terms = location_terms(location) if strict_city else []

    user_years = parse_user_experience(experience)
    type_word = JOB_TYPE_WORDS.get(job_type, "")
    level_word = "fresher" if (user_years == 0 and job_type != "Internship") else ""
    where = location if strict_city else "India"
    query = " ".join(p for p in [role, type_word, level_word, "in", where] if p)

    params = {"engine": "google_jobs", "q": query, "hl": "en", "gl": "in", "api_key": SERPAPI_KEY}
    if strict_city:
        params["location"] = location if "india" in location.lower() else f"{location}, India"
    else:
        params["location"] = "India"
    if job_type in JOB_TYPE_CHIPS:
        params["chips"] = f"employment_type:{JOB_TYPE_CHIPS[job_type]}"

    collected, loose, seen, raw_total = [], [], set(), 0
    removed_type = removed_exp = removed_loc = 0
    for _ in range(max_pages):
        data, err = search_page(params)
        if err:
            if not collected:
                return [], err
            notes.append(err)
            break
        raw_list = data.get("jobs_results") or []
        raw_total += len(raw_list)
        chips_active = "chips" in params          # Google's own job-type filter is still applied
        for raw in raw_list:
            if not isinstance(raw, dict):
                continue
            try:
                job = normalize_job(raw)
            except Exception:
                continue
            if not location_ok(job["location"], city_terms, strict_city):
                removed_loc += 1
                continue
            # Google already filtered by type, so an unlabelled job here matches the chosen type
            if chips_active and job["job_type"].lower() in UNKNOWN_LABELS and job_type in JOB_TYPE_WORDS:
                job["job_type"] = f"{JOB_TYPE_WORDS[job_type].title()} (per Google filter)"
            match = job_type_match(job, job_type)
            if match == "no":
                removed_type += 1
                continue
            if not experience_ok(job, user_years):
                removed_exp += 1
                continue
            key = (job["title"].lower(), job["company"].lower(), job["location"].lower())
            if key in seen:
                continue
            seen.add(key)
            (collected if match == "yes" else loose).append(job)
        token = (data.get("serpapi_pagination") or {}).get("next_page_token")
        if len(collected) + len(loose) >= max_jobs * 2 or len(collected) >= max_jobs or not token:
            break
        params["next_page_token"] = token

    if not collected and not loose:
        if raw_total == 0:
            return [], ("No Google Jobs results were found. Try a broader role, a different job type, "
                        "or choose 'No Preferred Location'.")
        return [], (f"Google Jobs returned {raw_total} results, but none matched all your filters "
                    f"(location: {where}; job type: {job_type}; experience: {experience or 'any'}). "
                    f"{removed_loc} were removed for location, {removed_type} for job type and {removed_exp} for experience. "
                    "Try 'Any' job type, a broader role, or 'No Preferred Location'.")

    rank = lambda j: relevance_score(j, role, user_skills or [])
    best = sorted(collected, key=rank, reverse=True)[:max_jobs + 5]
    others = sorted(loose, key=rank, reverse=True)[:5]

    # Google's text often has no experience. Read it from the apply page, then filter again.
    n_read = enrich_experience(best + others)
    if n_read:
        notes.append(f"Experience for {n_read} job(s) was read from the employer's own page.")
    before = len(best) + len(others)
    best = [j for j in best if experience_ok(j, user_years)]
    others = [j for j in others if experience_ok(j, user_years)]
    removed_exp += before - len(best) - len(others)

    ranked = best[:max_jobs]
    extra = others[:max_jobs - len(ranked)]
    if extra:
        notes.append(f"{len(extra)} job(s) at the end have no job type stated by the employer, "
                     f"so they could not be confirmed as '{job_type}'.")
    ranked = ranked + extra
    if not ranked:
        return [], (f"Jobs were found, but none matched your experience level ({experience or 'any'}). "
                    "Try a higher experience value or leave Experience empty.")
    if len(ranked) < max_jobs:
        notes.append(f"Only {len(ranked)} jobs matched your filters (removed: {removed_loc} for location, "
                     f"{removed_type} for job type, {removed_exp} for experience level).")
    return ranked, " ".join(notes)


print("✅ get_jobs() ready.")

✅ get_jobs() ready.


In [ ]:
# ============================================================
# CELL 5: Career analysis, skill gaps and ATS-style score
# ============================================================
ACTION_WORDS = ["developed", "built", "created", "implemented", "designed", "optimized",
                "analyzed", "improved", "automated", "deployed", "tested", "managed", "led"]

SECTION_PATTERNS = {
    "Summary / Objective": r"\b(summary|objective|profile|about me)\b",
    "Education": r"\beducation\b|\bacademic",
    "Skills": r"\bskills?\b",
    "Projects": r"\bprojects?\b",
    "Experience / Internship": r"\b(experience|internship|internships|work history|employment)\b",
    "Certifications": r"\bcertifications?\b|\bcertificates?\b|\bcertified\b",
    "Achievements": r"\b(achievements?|awards?|accomplishments?|honou?rs)\b",
    "GitHub / LinkedIn / Portfolio": r"github\.com|linkedin\.com|\bportfolio\b",
}


def analyze_skill_gap(user_skills, jobs):
    """Compare user skills with the skills required by the retrieved jobs."""
    counts = Counter()
    for job in jobs:
        counts.update(set(detect_skills(job_text(job))))

    common = counts.most_common()
    strong = [s for s, c in common if c >= 2]          # skills asked by at least 2 jobs
    required = strong[:15] if len(strong) >= 5 else [s for s, _ in common[:12]]

    user_set = set(user_skills)
    matched = [s for s in required if s in user_set]
    missing = [s for s in required if s not in user_set]
    percent = round(100 * len(matched) / len(required)) if required else 0
    return {"required": required, "matched": matched, "missing": missing,
            "percent": percent, "counts": dict(counts)}


def get_learning_recommendations(missing_skills):
    """Syllabus-relevant topic for each missing skill (learning only, not 'add to resume')."""
    return [(s, COURSE_MAP.get(s, "Related topic in your CSE / AI-ML syllabus")) for s in missing_skills]


def calculate_ats_score(resume_text, gap):
    """Academic heuristic: keyword match (40%), sections (30%), contact (10%), action words (20%)."""
    low = (resume_text or "").lower()
    resume_skills = detect_skills(resume_text)

    required = gap["required"]
    keyword = round(100 * len(set(resume_skills) & set(required)) / len(required)) if required else 0

    sections_found = [name for name, pat in SECTION_PATTERNS.items() if re.search(pat, low)]
    sections_missing = [name for name in SECTION_PATTERNS if name not in sections_found]
    sections = round(100 * len(sections_found) / len(SECTION_PATTERNS))

    has_email = bool(re.search(r"[\w.+-]+@[\w-]+\.[\w.-]+", resume_text or ""))
    phones = re.findall(r"\+?\d[\d\s\-()]{8,}\d", resume_text or "")
    has_phone = any(len(re.sub(r"\D", "", p)) >= 10 for p in phones)
    contact = (50 if has_email else 0) + (50 if has_phone else 0)

    action_found = [w for w in ACTION_WORDS if re.search(r"\b" + w + r"\b", low)]
    action = min(100, round(100 * len(action_found) / 8))      # 8 different action words = full marks

    overall = round(0.40 * keyword + 0.30 * sections + 0.10 * contact + 0.20 * action)
    return {"overall": overall, "keyword": keyword, "sections": sections, "contact": contact,
            "action": action, "sections_found": sections_found, "sections_missing": sections_missing,
            "action_found": action_found, "has_email": has_email, "has_phone": has_phone,
            "resume_skills": resume_skills}


print("✅ Analysis functions ready.")

✅ Analysis functions ready.


In [ ]:
# ============================================================
# CELL 6: Detailed (rule-based) resume improvement suggestions
# ============================================================
WEAK_PHRASES = ["worked on", "responsible for", "helped", "involved in", "participated in",
                "did ", "made ", "assisted"]
UNNECESSARY = ["date of birth", "dob", "father", "mother", "marital status", "religion",
               "nationality", "hobbies", "passport", "declaration", "gender"]
WARN, OK = "⚠️ Needs improvement", "✅ Looks fine – small polish"
TEMPLATE_NOTE = "(Example template – replace the [brackets] with your own real details. Do not add anything you have not actually done.)"


def quote(text):
    return "\n".join("> " + line for line in str(text).split("\n"))


def generate_resume_suggestions(resume_text, user_skills, role, education, jobs, gap, ats):
    text = resume_text or ""
    low = text.lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    found, total = ats["sections_found"], len(jobs)
    counts = gap["counts"]
    # most job-relevant skills first
    skills_disp = [display_skill(s) for s in sorted(user_skills, key=lambda x: -counts.get(x, 0))]
    top3 = ", ".join(skills_disp[:3]) if skills_disp else "[your skills]"
    role_disp = role.strip() if role and role.strip() else "[target role]"
    req = gap["required"]
    missing = gap["missing"]
    req_text = ", ".join(display_skill(s) for s in req[:6]) or "the skills listed in the job descriptions"

    def rel(skills):
        parts = [f"{display_skill(s)} appears in {counts[s]} of {total} retrieved jobs"
                 for s in skills[:4] if counts.get(s)]
        return "; ".join(parts) if parts else f"The retrieved jobs commonly ask for: {req_text}."

    weak_lines = [l for l in lines if any(l.lower().lstrip("-•*● ").startswith(p) for p in WEAK_PHRASES)]
    words = len(text.split())
    quant_hits = len(re.findall(r"\d+(?:\.\d+)?\s*%|\b\d+\+?\s*(?:users|projects|records|datasets|students|models|apis|features|samples|rows|endpoints)\b", low))
    unnecessary_found = [u for u in UNNECESSARY if re.search(r"\b" + re.escape(u) + r"\b", low)]
    matched_req = [s for s in req if s in ats["resume_skills"]]
    S = []

    def add(title, ok, current, suggested, why, relevance):
        S.append({"title": title, "status": OK if ok else WARN, "current": current,
                  "suggested": suggested, "why": why, "relevance": relevance})

    # 1 Summary
    has_summary = "Summary / Objective" in found
    add("Resume Summary", has_summary,
        "Your resume already has a summary/objective section." if has_summary
        else "No summary or objective section was found.",
        f"Computer Science student with hands-on knowledge of {top3}, looking for a {role_disp} "
        f"opportunity to apply my skills to real-world projects and keep learning.\n{TEMPLATE_NOTE}",
        "The summary is read first by recruiters and ATS tools. It should state who you are, your key skills and your target role.",
        rel(user_skills))

    # 2 Target role headline
    role_in = bool(role and role.strip().lower() in low)
    add("Career Objective / Target Role Headline", role_in,
        f"The target role '{role_disp}' appears in your resume." if role_in
        else f"The target role '{role_disp}' is not mentioned anywhere in your resume.",
        f"{role_disp} | {top3}\n(Place this as a headline under your name.)",
        "Using the same role title as the job posting helps ATS keyword matching and shows clear career direction.",
        f"The {total} retrieved jobs were searched using the role '{role_disp}'.")

    # 3 Skills section
    layout = []
    for cat, members in SKILL_CATEGORIES.items():
        mine = [display_skill(s) for s in members if s in user_skills]
        if mine:
            layout.append(f"{cat}: " + ", ".join(mine))
    has_skills = "Skills" in found
    add("Skills Section", has_skills,
        "A skills section exists. Check that it is grouped clearly." if has_skills
        else "No clearly labelled Skills section was found.",
        ("\n".join(layout) if layout else "Programming Languages: [..]\nDatabases: [..]\nTools: [..]")
        + "\n(Built only from skills detected in your resume/profile.)",
        "Grouped skills are easier for recruiters to scan and for ATS to parse.",
        rel(matched_req or user_skills))

    # 4 Job-specific keywords
    if missing:
        first = display_skill(missing[0])
        sug = (f"Learn first: " + ", ".join(f"{display_skill(s)} ({COURSE_MAP.get(s, 'related syllabus topic')})"
                                            for s in missing[:4])
               + f".\nOnly after you have really studied/used a skill, add a line such as: 'Used {first} to [what you actually built or practised]'.")
    else:
        sug = "You already cover the most requested keywords. Make sure each appears in a project or experience bullet, not only in the skills list."
    add("Job-Specific Keywords", not missing,
        f"Resume has {len(matched_req)} of {len(req)} most requested job skills. "
        + ("Not found: " + ", ".join(display_skill(s) for s in missing[:8]) + "." if missing else "All found."),
        sug,
        "ATS tools compare resume keywords with the job description. Add keywords only when they are true.",
        rel(missing or matched_req))

    # 5 Projects section
    has_proj = "Projects" in found
    add("Projects Section", has_proj,
        "A Projects section exists." if has_proj else "No Projects section was found.",
        f"Projects\n[Project Name] – [Tech used, e.g. {top3}]\n- Developed [what you built] that [problem it solves].\n- Implemented [feature] using [technology].\n{TEMPLATE_NOTE}",
        "Projects are the strongest evidence of skill for students and freshers.",
        rel(user_skills))

    # 6 Project descriptions
    add("Project Descriptions", not weak_lines,
        f"Weak wording found, for example: \"{weak_lines[0]}\"" if weak_lines
        else "No obviously weak project wording was detected. Check each bullet answers: what, how, result.",
        f"Developed a [type of project] using {top3} that [what it does / result].\n{TEMPLATE_NOTE}",
        "Vague statements like 'worked on a project' do not show what you did or which technologies you used.",
        rel(user_skills))

    # 7 Experience / internship
    has_exp = "Experience / Internship" in found
    add("Experience / Internship", has_exp,
        "An experience/internship section exists." if has_exp
        else "No experience or internship section found.",
        "[Role] – [Organization], [Duration]\n- Developed / Implemented [task] using [technology], resulting in [outcome].\n"
        "If you have none, add a 'Training / Virtual Internship / Open-source / Hackathon' section – only with real activities.",
        "Even short internships, virtual internships or training programs show practical exposure.",
        f"Many of the {total} jobs are matched against practical experience; internships are common for entry-level roles.")

    # 8 Education
    edu_detail = bool(re.search(r"cgpa|gpa|percentage|\d{2}(?:\.\d+)?\s*%", low)) and bool(re.search(r"20\d\d", low))
    edu_example = education.strip() if education and education.strip() else "[Degree], [College], [Start–End year], CGPA: [x.xx]"
    add("Education", edu_detail,
        "Education includes marks and year." if edu_detail
        else "Education details may be missing marks (CGPA/%) or graduation year.",
        f"{edu_example}\nAdd relevant coursework such as Data Structures, DBMS, Machine Learning if you studied them.",
        "Recruiters check degree, year and CGPA quickly; relevant coursework adds keywords.",
        "Fresher and internship roles often have an education requirement in the job description.")

    # 9 Certifications
    has_cert = "Certifications" in found
    cert_topics = ", ".join(sorted({COURSE_MAP.get(s, "") for s in missing[:4] if COURSE_MAP.get(s)})) or "your core syllabus subjects"
    add("Certifications", has_cert,
        "A certifications section exists." if has_cert else "No certifications section was found.",
        f"Consider certified courses in: {cert_topics}. Add each one only after completing it:\n[Course Name] – [Platform], [Year]",
        "Certificates support skills you are still building and show initiative.",
        rel(missing))

    # 10 Achievements
    has_ach = "Achievements" in found
    add("Achievements", has_ach,
        "An achievements section exists." if has_ach else "No achievements section was found.",
        "Achievements\n- [Hackathon / coding contest / academic award] – [Rank or result], [Year]\n"
        "(Include only real achievements.)",
        "Achievements differentiate you from other freshers with similar skills.",
        "Employers for the retrieved roles value problem-solving and initiative.")

    # 11 GitHub / LinkedIn
    has_links = "GitHub / LinkedIn / Portfolio" in found
    add("GitHub / LinkedIn / Portfolio", has_links,
        "A GitHub/LinkedIn/portfolio link was found." if has_links
        else "No GitHub, LinkedIn or portfolio link found.",
        "LinkedIn: linkedin.com/in/[your-id]  |  GitHub: github.com/[your-username]\n"
        "Make sure your GitHub has the projects listed on your resume with a README.",
        "Links let recruiters verify your work.",
        rel(["git", "github"] + user_skills))

    # 12 Action verbs
    n_act = len(ats["action_found"])
    cur12 = (f"Only {n_act} action verb(s) detected: " + (", ".join(ats["action_found"]) or "none") + ". "
             + (f"Weak phrasing example: \"{weak_lines[0]}\"" if weak_lines else ""))
    add("Action Verbs", n_act >= 6, cur12.strip(),
        "Replace phrases such as 'worked on' or 'responsible for' with verbs like: Developed, Built, Implemented, Designed, Optimized, Automated, Deployed, Analyzed.",
        "Strong verbs show ownership and are scanned by ATS tools.",
        "Job descriptions use verbs like develop, build, design and implement.")

    # 13 Quantifiable achievements
    add("Quantifiable Achievements", quant_hits >= 2,
        f"Only {quant_hits} measurable result(s) detected.",
        "Improved [metric] by [X]% by [action]. / Built a model with [X]% accuracy on [dataset]. / Handled [N] records or users.\n"
        "(Use only real numbers from your projects.)",
        "Numbers make impact concrete and credible.",
        "Employers prefer candidates who can show measurable results.")

    # 14 Formatting
    problems = []
    if words < 250:
        problems.append(f"the resume looks short ({words} words)")
    if words > 900:
        problems.append(f"the resume looks long ({words} words)")
    if not ats["has_email"]:
        problems.append("no email address found")
    if not ats["has_phone"]:
        problems.append("no phone number found")
    add("Formatting & Contact Details", not problems,
        ("Issues: " + "; ".join(problems) + ".") if problems else "Length and contact details look reasonable.",
        "Use one column, standard headings (Summary, Education, Skills, Projects, Experience), no tables, images or text boxes, "
        "and keep to 1 page for freshers. Put email and phone at the top.",
        "Simple layouts are parsed more reliably by ATS software.",
        "Clean, standard formatting lets your matching skills be read correctly.")

    # 15 Unnecessary information
    add("Remove Unnecessary Information", not unnecessary_found,
        ("Found: " + ", ".join(unnecessary_found) + ".") if unnecessary_found
        else "No unnecessary personal details were detected.",
        "Remove date of birth, parents' names, marital status, religion and photo. Keep hobbies only if short and relevant."
        if unnecessary_found else "Keep the resume focused on skills, projects and education.",
        "The space is better used for skills and projects.",
        "Recruiters for the retrieved jobs focus on skills and experience.")
    return S


def format_suggestions(suggestions):
    out = []
    for i, s in enumerate(suggestions, 1):
        out.append(f"### {i}. {s['title']}  —  {s['status']}\n")
        out.append(f"**Current:**\n\n{quote(s['current'])}\n")
        out.append(f"**Suggested Replacement:**\n\n{quote(s['suggested'])}\n")
        out.append(f"**Why:** {s['why']}\n")
        out.append(f"**Job Relevance:** {s['relevance']}\n")
        out.append("---\n")
    return "\n".join(out)


print("✅ Resume suggestion functions ready.")

✅ Resume suggestion functions ready.


In [ ]:
# ============================================================
# CELL 7: Main analyze_career() function
# ============================================================
def format_job_card(i, job):
    desc = re.sub(r"\s+", " ", job["job_description"]).strip()
    if len(desc) > 450:
        desc = desc[:450].rsplit(" ", 1)[0] + "..."
    if job["apply_link"]:
        link_line = f"🔗 [Apply for this Job]({job['apply_link']})"
    else:
        link_line = "🔗 Apply for this Job: Application link not provided."
    src = f"  |  [View on Google Jobs]({job['source_link']})" if job["source_link"] else ""
    return (f"### {i}. {job['title']}\n\n"
            f"**Company:** {job['company']}  \n"
            f"**Location:** {job['location']}  \n"
            f"**Job Type:** {job['job_type']}  \n"
            f"**Experience:** {job['experience']}  \n"
            f"**Source:** {job['via']}\n\n"
            f"**Description:** {desc}\n\n"
            f"{link_line}{src}\n\n---\n")


def _analyze_career(profile_method, manual_skills, resume_file, education, experience,
                    role, location, location_pref, job_type):
    use_resume = profile_method == "Upload Resume"
    experience_input = (experience or "").strip()      # used for filtering jobs (before any default text)
    resume_text = ""
    notes = []

    # ---- 1. Build the profile ----
    if use_resume:
        path = resume_file if isinstance(resume_file, str) else getattr(resume_file, "name", None)
        if not path:
            return "⚠️ Please upload your resume as a PDF."
        resume_text = extract_resume_text(path)
        if not resume_text.strip():
            return ("⚠️ No text could be read from this PDF. It may be scanned or image-only. "
                    "Please upload a **text-based PDF** (exported from Word/Google Docs, not a photo or scan).")
        user_skills = detect_skills(resume_text)
        if not user_skills:
            notes.append("⚠️ No skills from the skill list were detected in your resume. "
                         "Jobs are still searched using your target role.")
        if not (education or "").strip():
            education = detect_education(resume_text)
        if not (experience or "").strip():
            experience = "Mentioned in resume" if re.search(r"internship|experience", resume_text, re.I) else "Fresher / Not specified"
    else:
        user_skills = parse_manual_skills(manual_skills)
        if not user_skills:
            return "⚠️ Please enter at least one skill (separate with commas, semicolons or new lines)."

    role = (role or "").strip()
    if not role:
        return "⚠️ Please enter a target job role (for example: Python Developer)."
    education = (education or "").strip() or "Not specified"
    experience = (experience or "").strip() or "Not specified"
    strict = location_pref == "Preferred Location Only" and (location or "").strip()
    loc_display = location.strip() if strict else "India (no preferred location)"

    profile = ["## 👤 Candidate Profile\n",
               f"**Skills {'detected' if use_resume else 'entered'}:** "
               + (", ".join(display_skill(s) for s in user_skills) or "None detected"),
               f"**Education:** {education}", f"**Experience:** {experience}",
               f"**Target Role:** {role}", f"**Preferred Location:** {loc_display}",
               f"**Job Type:** {job_type or 'Any'}\n"]
    out = ["  \n".join(profile)] + notes

    # ---- 2. Search jobs ----
    jobs, message = get_jobs(role, location, location_pref, job_type, user_skills, experience_input)
    if not jobs:
        out.append(f"## 🔎 Jobs\n\n{message}")
        return "\n\n".join(out)
    filters = f"Job Type = {job_type or 'Any'}"
    filters += (f"; Experience = {experience_input} (jobs needing more experience removed)"
                if parse_user_experience(experience_input) is not None else "; Experience = no filter")
    out.append(f"🎯 **Filters applied:** {filters}")
    if message:
        out.append(f"ℹ️ {message}")

    # ---- 3. Skill gap ----
    gap = analyze_skill_gap(user_skills, jobs)

    # ---- 4. ATS score (resume only) ----
    ats = None
    if use_resume:
        ats = calculate_ats_score(resume_text, gap)
        out.append(
            "## 📊 Resume ATS-Style Compatibility Score\n\n"
            f"# {ats['overall']}%\n\n"
            f"- **Keyword Match:** {ats['keyword']}%\n"
            f"- **Resume Sections:** {ats['sections']}% "
            f"(missing: {', '.join(ats['sections_missing']) or 'none'})\n"
            f"- **Contact Information:** {ats['contact']}%\n"
            f"- **Action Words:** {ats['action']}% ({len(ats['action_found'])} found)\n\n"
            "*This is an academic heuristic. It is not the score of any real company's ATS and "
            "it does not predict your chance of being shortlisted or hired.*")

    # ---- 5. Matching skills, gaps, learning ----
    matched = "\n".join(f"- {display_skill(s)}" for s in gap["matched"]) or "- None of the most requested skills found yet"
    missing = "\n".join(f"- {display_skill(s)}" for s in gap["missing"]) or "- No major gaps found 🎉"
    out.append(f"## ✅ Your Relevant Skills\n\n{matched}")
    out.append(f"## ❌ Skill Gaps\n\n{missing}\n\n**Skill Match:** {gap['percent']}% "
               f"({len(gap['matched'])} of {len(gap['required'])} frequently requested skills)")
    recs = get_learning_recommendations(gap["missing"][:8])
    if recs:
        lines = "\n".join(f"- **{display_skill(s)}** → {course}" for s, course in recs)
        out.append(f"## 📚 Recommended Learning\n\n{lines}\n\n"
                   "*Learn these first. Add a skill to your resume only after you have actually studied or used it.*")

    # ---- 6. Resume suggestions ----
    if use_resume:
        suggestions = generate_resume_suggestions(resume_text, user_skills, role, education, jobs, gap, ats)
        out.append(f"## ✍️ Resume Improvement Suggestions ({len(suggestions)})\n\n" + format_suggestions(suggestions))

    # ---- 7. Jobs ----
    cards = "\n".join(format_job_card(i, j) for i, j in enumerate(jobs, 1))
    out.append(f"## 💼 Relevant Job Opportunities ({len(jobs)})\n\n{cards}")
    return "\n\n".join(out)


def analyze_career(profile_method, manual_skills, resume_file, education, experience,
                   role, location, location_pref, job_type):
    """Called by Gradio. Never raises, so the app never crashes."""
    try:
        return _analyze_career(profile_method, manual_skills, resume_file, education, experience,
                               role, location, location_pref, job_type)
    except Exception as e:
        return f"❌ Something went wrong ({type(e).__name__}: {e}). Please check your inputs and try again."


print("✅ analyze_career() ready.")

✅ analyze_career() ready.


In [ ]:
# ============================================================
# CELL 8: Gradio interface (no API-key box)
# ============================================================
def toggle_profile_inputs(method):
    return (gr.update(visible=(method == "Enter Skills Manually")),
            gr.update(visible=(method == "Upload Resume")))


with gr.Blocks(title="CareerAI") as demo:
    gr.Markdown("# 🎓 CareerAI\n"
                "### An AI-Powered Platform for Job Matching, Resume Optimization and Skill Gap Analysis\n"
                "Live job search powered by Google Jobs (SerpApi).")
    with gr.Row():
        with gr.Column(scale=1):
            profile_method = gr.Radio(["Enter Skills Manually", "Upload Resume"],
                                      value="Enter Skills Manually", label="Profile Method")
            manual_skills = gr.Textbox(label="Skills", lines=3, visible=True,
                                       placeholder="Python, SQL, Git (comma, semicolon or new line separated)")
            resume_file = gr.File(label="Upload Resume PDF", file_types=[".pdf"],
                                  type="filepath", visible=False)
            education = gr.Textbox(label="Education", placeholder="B.Tech CSE")
            experience = gr.Textbox(label="Experience", placeholder="Fresher, 1 year, 2 years (used to filter jobs)")
            role = gr.Textbox(label="Target Job Role", placeholder="Python Developer")
            location = gr.Textbox(label="Preferred Location", placeholder="Hyderabad")
            location_pref = gr.Radio(["Preferred Location Only", "No Preferred Location"],
                                     value="Preferred Location Only", label="Location Preference")
            job_type = gr.Dropdown(["Any", "Full Time", "Part Time", "Internship", "Contract"],
                                   value="Any", label="Job Type")
            run_btn = gr.Button("🔎 Find Jobs & Analyze Career", variant="primary")
        with gr.Column(scale=2):
            gr.Markdown("## Career Analysis")
            output = gr.Markdown()

    profile_method.change(toggle_profile_inputs, inputs=profile_method,
                          outputs=[manual_skills, resume_file])
    run_btn.click(analyze_career,
                  inputs=[profile_method, manual_skills, resume_file, education, experience,
                          role, location, location_pref, job_type],
                  outputs=output)

print("✅ Gradio interface built. Run the next cell to launch it.")

✅ Gradio interface built. Run the next cell to launch it.


In [ ]:

# ============================================================
# CELL 9: Launch the app (creates the shareable public URL)
try:
    demo.close()          # close any earlier instance
except Exception:
    pass

demo.queue().launch(share=True, debug=True)

Closing server running on port: 7860
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5bae0fd6115c9dca60.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
